In [6]:
%pip install dash paho-mqtt

import json
import os
import threading
import time
from datetime import datetime, timezone

import paho.mqtt.client as mqtt
from dash import Dash, dcc, html, Input, Output, State, callback_context

MQTT_HOST = os.environ.get("MQTT_HOST", "localhost")
MQTT_PORT = int(os.environ.get("MQTT_PORT", "1883"))
ROOT_TOPIC = "cps/plant-1"
TELEMETRY_TOPIC = f"{ROOT_TOPIC}/telemetry"
COMMAND_TOPIC = f"{ROOT_TOPIC}/actuator/motor/set"
STATE_TOPIC = f"{ROOT_TOPIC}/actuator/motor/state"

latest = {"temperature_c": None, "pressure_kpa": None, "motor_on": None, "received_at": None, "connected": False, "error": None}
state_lock = threading.Lock()


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: C:\Users\iamwa\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
def on_connect(client, userdata, flags, reason_code, properties):
    with state_lock:
        latest["connected"] = (reason_code == 0)
        latest["error"] = None if reason_code == 0 else f"MQTT connection error: {reason_code}"
    if reason_code == 0:
        client.subscribe([(TELEMETRY_TOPIC, 0), (STATE_TOPIC, 1)])

def on_message(client, userdata, msg):
    try:
        data = json.loads(msg.payload.decode("utf-8"))
        with state_lock:
            if msg.topic == TELEMETRY_TOPIC:
                for key in ("temperature_c", "pressure_kpa"):
                    if key in data:
                        latest[key] = float(data[key])
                if "motor_on" in data:
                    latest["motor_on"] = bool(data["motor_on"])
            elif msg.topic == STATE_TOPIC and "motor_on" in data:
                latest["motor_on"] = bool(data["motor_on"])
            latest["received_at"] = datetime.now(timezone.utc).astimezone().strftime("%H:%M:%S %Z")
    except (UnicodeDecodeError, json.JSONDecodeError, TypeError, ValueError) as exc:
        with state_lock:
            latest["error"] = f"Invalid MQTT payload: {exc}"

mqtt_client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="cps-dashboard")
mqtt_client.on_connect = on_connect
mqtt_client.on_message = on_message
mqtt_client.connect(MQTT_HOST, MQTT_PORT, keepalive=30)
mqtt_client.loop_start()
print(f"Connecting to MQTT broker at {MQTT_HOST}:{MQTT_PORT}...")

Connecting to MQTT broker at localhost:1883...


In [8]:
RUN_SIMULATOR = True
simulator_stop = threading.Event()

def run_simulator():
    sim = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id="cps-simulator")
    motor = {"on": False}
    def connected(client, userdata, flags, reason_code, properties):
        if reason_code == 0:
            client.subscribe(COMMAND_TOPIC, qos=1)
    def command_received(client, userdata, msg):
        try:
            command = json.loads(msg.payload.decode("utf-8"))
            if command.get("command") in ("ON", "OFF"):
                motor["on"] = (command["command"] == "ON")
                client.publish(STATE_TOPIC, json.dumps({"motor_on": motor["on"], "updated_at": datetime.now(timezone.utc).isoformat()}), qos=1, retain=True)
        except (UnicodeDecodeError, json.JSONDecodeError):
            pass
    sim.on_connect = connected
    sim.on_message = command_received
    sim.connect(MQTT_HOST, MQTT_PORT, keepalive=30)
    sim.loop_start()
    try:
        while not simulator_stop.is_set():
            t = time.time()
            telemetry = {"temperature_c": round(24 + 2 * __import__("math").sin(t / 8), 2),
                         "pressure_kpa": round(101 + 0.7 * __import__("math").cos(t / 10), 2),
                         "motor_on": motor["on"]}
            sim.publish(TELEMETRY_TOPIC, json.dumps(telemetry), qos=0)
            simulator_stop.wait(1.0)
    finally:
        sim.loop_stop()
        sim.disconnect()

if RUN_SIMULATOR:
    simulator_thread = threading.Thread(target=run_simulator, daemon=True)
    simulator_thread.start()
    print("CPS simulator started.")

CPS simulator started.


In [9]:
app = Dash(__name__)
app.layout = html.Div([
    html.H2("CPS Live Monitor"),
    html.Div(id="connection", style={"marginBottom": "12px"}),
    html.Div([
        html.Div([html.H4("Temperature"), html.Div(id="temperature", style={"fontSize": "2rem"})], className="metric"),
        html.Div([html.H4("Pressure"), html.Div(id="pressure", style={"fontSize": "2rem"})], className="metric"),
        html.Div([html.H4("Motor feedback"), html.Div(id="motor-state", style={"fontSize": "2rem"})], className="metric"),
    ], style={"display": "flex", "gap": "24px", "flexWrap": "wrap"}),
    html.Div([
        html.Button("Motor ON", id="motor-on", n_clicks=0, style={"marginRight": "10px", "padding": "10px 18px"}),
        html.Button("Motor OFF", id="motor-off", n_clicks=0, style={"padding": "10px 18px"}),
    ], style={"marginTop": "22px"}),
    html.Div(id="command-result", style={"marginTop": "12px"}),
    html.Div(id="last-update", style={"marginTop": "8px", "color": "#666"}),
    dcc.Interval(id="refresh", interval=1000, n_intervals=0),
], style={"fontFamily": "Arial, sans-serif", "padding": "24px", "maxWidth": "900px", "margin": "auto"})

@app.callback(Output("connection", "children"), Output("temperature", "children"),
              Output("pressure", "children"), Output("motor-state", "children"),
              Output("last-update", "children"), Input("refresh", "n_intervals"))
def refresh_dashboard(_):
    with state_lock:
        data = dict(latest)
    connection = "Broker connected" if data["connected"] else (data["error"] or "Waiting for broker connection…")
    temperature = f"{data['temperature_c']:.1f} °C" if data["temperature_c"] is not None else "—"
    pressure = f"{data['pressure_kpa']:.1f} kPa" if data["pressure_kpa"] is not None else "—"
    motor = "ON" if data["motor_on"] is True else ("OFF" if data["motor_on"] is False else "Unknown")
    updated = f"Last message: {data['received_at']}" if data["received_at"] else "No telemetry received yet"
    return connection, temperature, pressure, motor, updated

@app.callback(Output("command-result", "children"), Input("motor-on", "n_clicks"),
              Input("motor-off", "n_clicks"), prevent_initial_call=True)
def send_motor_command(on_clicks, off_clicks):
    trigger = callback_context.triggered_id
    command = "ON" if trigger == "motor-on" else "OFF"
    payload = {"command": command, "requested_at": datetime.now(timezone.utc).isoformat()}
    try:
        info = mqtt_client.publish(COMMAND_TOPIC, json.dumps(payload), qos=1)
        info.wait_for_publish(timeout=3)
        return f"{command} request sent; waiting for actuator feedback."
    except Exception as exc:
        return f"Command publish failed: {exc}"

app.run(jupyter_mode="inline", debug=False, port=8050)

In [10]:
if RUN_SIMULATOR:
    simulator_stop.set()
    if 'simulator_thread' in globals():
        simulator_thread.join(timeout=3)
mqtt_client.loop_stop()
mqtt_client.disconnect()
print("MQTT clients stopped.")

MQTT clients stopped.
